In [1]:
import pandas as pd
import pprint
import requests
import pprint
import yaml


In [ ]:
%store -r cleaned_data
pd.set_option('display.max_columns', None,)

pprint.pprint(cleaned_data.columns.tolist())
# print(cleaned_data.columns.tolist())
display(cleaned_data)

# data_user_input_mode_confirm
# data_user_input_purpose_confirm
# data_user_input_replaced_mode
# Mode_confirm
# Replaced_mode
# Trip_purpose
# Mode


In [ ]:
# List of columns to move to the beginning
columns_to_move = [
    'Mode',
    'data_user_input_mode_confirm', # is what the user chooses
    'Mode_confirm',
    'data_user_input_replaced_mode',
    'Replaced_mode',
    'data_user_input_purpose_confirm',
    'Trip_purpose',
]

# Get the current columns of the DataFrame
current_columns = cleaned_data.columns.tolist()

# Create a new column order: move specified columns to the beginning, followed by the rest
new_column_order = columns_to_move + [col for col in current_columns if col not in columns_to_move]

# Reorder the DataFrame
cleaned_data = cleaned_data[new_column_order]

# Display the columns to confirm the reordering
# pprint.pprint(cleaned_data.columns.tolist())

# Display the cleaned data

display(cleaned_data.head(2))

In [ ]:
print(cleaned_data['Mode_confirm'].unique())
# Get unique values from the 'data_user_input_mode_confirm' column
unique_modes = cleaned_data['data_user_input_mode_confirm'].unique()

# Convert all unique values to strings
unique_modes_str = [str(mode) for mode in unique_modes]

# Sort the unique values by the length of the strings
sorted_modes = sorted(unique_modes_str, key=len)

# Print the sorted unique values
print(sorted_modes)

In [ ]:
# Get unique values from the 'data_user_input_mode_confirm' column
unique_modes = cleaned_data['data_user_input_mode_confirm'].unique()

# Convert all unique values to strings
unique_modes_str = [str(mode) for mode in unique_modes]

# Sort the unique values by the length of the strings
sorted_modes = sorted(unique_modes_str, key=len)

# Print the sorted unique values
print(sorted_modes)

In [5]:
# Load API keys from YAML file
def load_api_keys(file_path="apikey.yaml"):
    with open(file_path, 'r') as file:
        config = yaml.safe_load(file)
    return config['api_keys']['ORS_API_KEY'], config['api_keys']['GOOGLE_API_KEY']

# Load the keys
ORS_API_KEY, GOOGLE_API_KEY = load_api_keys()


In [ ]:

# Coordinates for start and end (latitude,longitude)
start = '-82.382264,29.626036'  # Example: Coordinates for start (lon, lat for ORS)
# gainesville aldi. longitude , latitude
end = '-82.371415,29.604212'    # Example: Coordinates for end (lon, lat for ORS)
# gainesville mcdonalds, longitude , latitude

# Function to reverse geocode coordinates using OpenRouteService
def get_location(lat, lon):
    url = f'https://api.openrouteservice.org/geocode/reverse?api_key={ORS_API_KEY}&point.lon={lon}&point.lat={lat}'
    response = requests.get(url, verify=False)  # Disable SSL verification
    data = response.json()

    # Extract relevant address components
    address = data['features'][0]['properties']
    city = address.get('locality', 'N/A')
    state = address.get('region', 'N/A')
    country = address.get('country', 'N/A')

    return city, state, country

# Function to get travel time for a specific mode using OpenRouteService
def get_travel_time_ors(mode):
    url = f'https://api.openrouteservice.org/v2/directions/{mode}?api_key={ORS_API_KEY}&start={start}&end={end}'
    
    # Make a request
    response = requests.get(url, verify=False)  # Disable SSL verification
    data = response.json()

    # Extract the route summary
    route_summary = data['features'][0]['properties']['summary']
    total_distance = route_summary['distance']  # Distance in meters
    total_duration = route_summary['duration']  # Duration in seconds

    # Convert to more readable formats
    total_distance_km = total_distance / 1000  # Convert to kilometers
    total_duration_minutes = total_duration / 60  # Convert to minutes

    print(f"Mode: {mode}")
    print(f"Total distance: {total_distance_km:.2f} km")
    print(f"Total duration: {total_duration_minutes:.2f} minutes")
    print("-" * 30)

# Function to get travel time for transit using Google Distance Matrix API
def get_travel_time_google_transit():
    # Swap the coordinates to (latitude,longitude) for Google API
    start_lat, start_lon = map(str.strip, start.split(','))
    end_lat, end_lon = map(str.strip, end.split(','))
    
    # Construct Google Distance Matrix API URL
    google_start = f"{start_lon},{start_lat}"  # Swap to lat,lon format for Google
    google_end = f"{end_lon},{end_lat}"        # Swap to lat,lon format for Google
    
    url = f'https://maps.googleapis.com/maps/api/distancematrix/json?origins={google_start}&destinations={google_end}&mode=transit&key={GOOGLE_API_KEY}'
    
    # Make the request
    response = requests.get(url, verify=False)  # Disable SSL verification
    data = response.json()
    
    print(url, ':))))')

    if data['status'] == 'OK':
        row = data['rows'][0]['elements'][0]
        if row['status'] == 'OK':
            duration = row['duration']['text']
            distance = row['distance']['text']
            print(f"Mode: public transit")
            print(f"Total distance: {distance}")
            print(f"Total duration: {duration}")
            print("-" * 30)
        else:
            print(f"Error in transit response: {row['status']}")
    else:
        print(f"Error: {data['status']}")

# Function to print locations for start and end points
def print_locations():
    start_lat, start_lon = map(float, start.split(','))
    end_lat, end_lon = map(float, end.split(','))

    # Swap the order for the geocoding API (it needs lon,lat)
    start_city, start_state, start_country = get_location(start_lon, start_lat)
    print(f"Start location: {start_city}, {start_state}, {start_country}")

    end_city, end_state, end_country = get_location(end_lon, end_lat)
    print(f"End location: {end_city}, {end_state}, {end_country}")
    print("=" * 30)

# Print the locations for start and end points
print_locations()

# Get travel times for driving, cycling, and walking using OpenRouteService
modes = ['driving-car', 'cycling-regular', 'cycling-mountain', 'cycling-electric', 'foot-walking']
for mode in modes:
    get_travel_time_ors(mode)

# Get travel time for public transit using Google Distance Matrix API

# get_travel_time_google_transit()


In [7]:
modes_of_ors_transport = [
    "Driving-car",
    "Driving-hgv",
    "Cycling-regular",
    "Cycling-road",
    "Cycling-mountain",
    "Cycling-electric",
    "Foot-walking",
    "Foot-hiking",
    "Wheelchair"
]

In [8]:

# https://github.com/e-mission/em-public-dashboard/issues/31#issuecomment-1015675164

# perhaps store the cost for each year, as we have data across years;
# or simply get the inflation rate to convert between years.

# average car https://newsroom.aaa.com/wp-content/uploads/2022/08/2022-YourDrivingCosts-FactSheet-7-1.pdf
costs_dictionary = {
    'Car': 0.67, #https://www.gsa.gov/travel/plan-a-trip/transportation-airfare-rates-pov-rates-etc/privately-owned-vehicle-pov-mileage-reimbursement 
    'Transit': 0.855,
    'Ridehail': 2.5, #https://doi.org/10.1016/j.tra.2019.09.056
    'Walk': 0, 
    # temp value
    'Personal Micromobility': 0.3,
    'E-bike': 0.3,
    'Other': 0.3,
    
}
costs_dictionary['Shared Car'] = costs_dictionary['Car'] / 2
costs_dictionary['Shared Micromobility'] = costs_dictionary['Personal Micromobility'] / 2


In [9]:
# rahul used VTPI
# https://www.vtpi.org/tca/tca0501.pdf
mode_cost_per_mile = {
    # bicycle/skateboard
    'p_micro': 0.,
    'no_trip': 0.,
    # Shared car is half the cost of regular car, which is $0.6/mile.
    's_car': 0.3,
    'car': 0.6,
    # Average of bus and train taken.
    'transit': 0.5,
    # Shared bicyle or scooter - values taken from https://nacto.org/shared-micromobility-2020-2021/ and 
    # https://www.mckinsey.com/industries/automotive-and-assembly/our-insights/how-sharing-the-road-is-likely-to-transform-american-mobility
    's_micro': 0.3,
    # uber/taxi/lyft
    'ridehail': 2.,
    'walk': 0.,
    'unknown': 0.
}

# Assumptions.
mode_init_cost = {
    'p_micro': 0.,
    'no_trip': 0.,
    # Shared car is half the cost of regular car, which is $0.6/mile.
    's_car': 0.,
    # Rental car.
    'car': 0.,
    # Average of bus and train taken.
    'transit': 0.,
    # $1 unlocking cost.
    's_micro': 1.,
    # uber/taxi/lyft
    'ridehail': 1.5,
    'walk': 0.,
    'unknown': 0.
}